In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import model_from_json


In [2]:
print("Files in folder:", os.listdir())
print("model.json exists:", os.path.exists("model.json"))
print("best_model1.h5 exists:", os.path.exists("best_model1.h5"))


Files in folder: ['.ipynb_checkpoints', '.venv', '.vscode', '720p50_mobcal_ter.y4m', 'app.py', 'best_model1.h5', 'environment.yml', 'ffmpeg.exe', 'images', 'LICENSE.txt', 'logs', 'model.json', 'normal.mp4', 'pages', 'README.md', 'saved_models', 'testing_model.ipynb', 'test_frames', 'test_video.py', 'train.ipynb', 'video_dataset', 'video_model.keras', 'video_violence_model.h5']
model.json exists: True
best_model1.h5 exists: True


In [3]:
with open("model.json", "r") as f:
    model_json = f.read()

loaded_model = model_from_json(model_json)
loaded_model.load_weights("best_model1.h5")

print("✅ Model loaded successfully")
loaded_model.summary()


✅ Model loaded successfully
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 time_distributed (TimeDistr  (None, 8, 112, 112, 64)  1792      
 ibuted)                                                         
                                                                 
 time_distributed_1 (TimeDis  (None, 8, 56, 56, 64)    36928     
 tributed)                                                       
                                                                 
 time_distributed_2 (TimeDis  (None, 8, 28, 28, 64)    0         
 tributed)                                                       
                                                                 
 time_distributed_3 (TimeDis  (None, 8, 14, 14, 128)   73856     
 tributed)                                                       
                                                                 
 time_distributed_4 (TimeDis

In [4]:
loaded_model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 time_distributed (TimeDistr  (None, 8, 112, 112, 64)  1792      
 ibuted)                                                         
                                                                 
 time_distributed_1 (TimeDis  (None, 8, 56, 56, 64)    36928     
 tributed)                                                       
                                                                 
 time_distributed_2 (TimeDis  (None, 8, 28, 28, 64)    0         
 tributed)                                                       
                                                                 
 time_distributed_3 (TimeDis  (None, 8, 14, 14, 128)   73856     
 tributed)                                                       
                                                                 
 time_distributed_4 (TimeDis  (None, 8, 7, 7, 128)     0

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, Conv2D, MaxPooling2D, Flatten, LSTM, Dense, Dropout

SEQ_LEN = 8

seq_model = Sequential([
    TimeDistributed(Conv2D(64,(3,3), padding="same", strides=(2,2), activation="relu"),
                   input_shape=(SEQ_LEN,224,224,3)),
    TimeDistributed(Conv2D(64,(3,3), padding="same", strides=(2,2), activation="relu")),
    TimeDistributed(MaxPooling2D((2,2), strides=(2,2))),

    TimeDistributed(Conv2D(128,(3,3), padding="same", strides=(2,2), activation="relu")),
    TimeDistributed(MaxPooling2D((2,2), strides=(2,2))),
    TimeDistributed(Flatten()),

    LSTM(512, activation="relu", return_sequences=True),
    Dropout(0.5),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(5, activation="softmax")
])

seq_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
seq_model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 time_distributed (TimeDistr  (None, 8, 112, 112, 64)  1792      
 ibuted)                                                         
                                                                 
 time_distributed_1 (TimeDis  (None, 8, 56, 56, 64)    36928     
 tributed)                                                       
                                                                 
 time_distributed_2 (TimeDis  (None, 8, 28, 28, 64)    0         
 tributed)                                                       
                                                                 
 time_distributed_3 (TimeDis  (None, 8, 14, 14, 128)   73856     
 tributed)                                                       
                                                                 
 time_distributed_4 (TimeDis  (None, 8, 7, 7, 128)     0

In [6]:
class_names = ["blast", "blood", "fight", "gunshot", "normal"]
SEQ_LEN = 8
IMG_SIZE = 224

video_path = r"test_vid/normal.mpg"   # ✅ change to your video file path
cap = cv2.VideoCapture(video_path)

frames = []
last_label = "..."

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # preprocess frame
    f = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    f = f.astype("float32") / 255.0
    frames.append(f)

    if len(frames) == SEQ_LEN:
        sequence = np.expand_dims(np.array(frames, dtype="float32"), axis=0)  # (1,8,224,224,3)

        pred = loaded_model.predict(sequence, verbose=0)

        # if model returns per-frame probs: (1,8,5)
        if pred.ndim == 3:
            pred = pred.mean(axis=1)  # (1,5)

        class_id = int(np.argmax(pred, axis=1)[0])
        conf = float(np.max(pred))
        last_label = f"{class_names[class_id]} ({conf:.2f})"

        frames = []  # reset

    # display
    show = frame.copy()
    cv2.putText(show, last_label, (30, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow("Cartoon Violence Detection", show)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
